# Protein ↔ Disease Relation-Wise Merge

Merges Protein–Disease triples from CKG (×2), CrossBAR (×2), and DtiNet;
assigns `tail_id_is` by disease ID prefix (DOID vs MESH); fills protein head names
from UniProt; deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [2]:
import os
import pandas as pd
import numpy as np
import re

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PROTEIN_DISEASE/ALL_PROTEIN_DISEASE.parquet'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Protein Lookup Dictionaries — UniProt

In [3]:
# RecName dict (for sources that store UniProt ACs in head)
Uniprot_Recname = pd.read_csv(DB_DIR + 'uniprot/Uniprot_ID_Recname.csv')
Uniprot_Recname_dict = dict(zip(Uniprot_Recname['AC'], Uniprot_Recname['RecName']))

# Parsed TrEMBL dict (AC → NAME) for head-name fallback
uniprot = pd.read_csv(DB_DIR + 'uniprot/uniprot_parsed_results.csv')
uniprot_trembl_AC_Name_dict = dict(zip(uniprot['AC'], uniprot['NAME']))
del uniprot

print(f"UniProt RecName dict: {len(Uniprot_Recname_dict):,} | TrEMBL dict: {len(uniprot_trembl_AC_Name_dict):,}")

UniProt RecName dict: 794,151 | TrEMBL dict: 252,635,149


## 2. Load KG Sources

### 2a. CKG (×2)

In [4]:
CKG_Protein_Disease1 = pd.read_csv(PROC_DIR + 'CKG/File_21_protein_Disease_CKG.csv')
CKG_Protein_Disease1.columns = CKG_Protein_Disease1.columns.str.lower()
CKG_Protein_Disease1.rename(columns={'tail_do_name': 'tail_detail_name', 'head_alt_name': 'head_detail_name'}, inplace=True)

CKG_Protein_Disease2 = pd.read_csv(PROC_DIR + 'CKG/File_25_protein_disease_CKG.csv')
CKG_Protein_Disease2.columns = CKG_Protein_Disease2.columns.str.lower()
CKG_Protein_Disease2.rename(columns={'tail_do_name': 'tail_detail_name', 'head_alt_name': 'head_detail_name'}, inplace=True)

print(f"CKG 1: {len(CKG_Protein_Disease1):,} rows")
print(f"CKG 2: {len(CKG_Protein_Disease2):,} rows")

CKG 1: 30,241,580 rows
CKG 2: 927,590 rows


In [5]:
CKG_Protein_Disease1['kg_type'] = 'Generalised'
CKG_Protein_Disease1['kg_source'] = 'CKG'
CKG_Protein_Disease1['species'] = 'Homo species'
CKG_Protein_Disease1

,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,tail_do_alt_ids,head_id_is,tail_id_is,evidence_type,score,kg_type,species
0,Q9UCA1,ASSOCIATED_WITH_INTEGRATED,DOID:10605,Protein,Disease,DISEASES,CKG,Thrombin heavy chain,short bowel syndrome,MESH:D012778|NCI:C99059|SNOMEDCT_US_2023_03_01...,Uniprot_ID,Disease_Ontology,compiled,1.657,Generalised,Homo species
1,Q8N425,ASSOCIATED_WITH_INTEGRATED,DOID:654,Protein,Disease,DISEASES,CKG,(E3-independent) E2 ubiquitin-conjugating enzyme,overnutrition,MESH:D044343|SNOMEDCT_US_2023_03_01:302872003|...,Uniprot_ID,Disease_Ontology,compiled,0.982,Generalised,Homo species
2,Q86UJ9,ASSOCIATED_WITH_INTEGRATED,DOID:0060037,Protein,Disease,DISEASES,CKG,Cyclic AMP-responsive element-binding protein 5,developmental disorder of mental health,DOID:0060037,Uniprot_ID,Disease_Ontology,compiled,1.840,Generalised,Homo species
3,Q7Z639,ASSOCIATED_WITH_INTEGRATED,DOID:1240,Protein,Disease,DISEASES,CKG,Ubiquitin conjugation factor E4 A {ECO:0000305},leukemia,ICD10CM:C95.90|ICD9CM:208|ICDO:9800/3|MESH:D00...,Uniprot_ID,Disease_Ontology,compiled,1.128,Generalised,Homo species
4,Q1P9S9,ASSOCIATED_WITH_INTEGRATED,DOID:0050328,Protein,Disease,DISEASES,CKG,Arresten,congenital hypothyroidism,GARD:1487|ICD10CM:E00.1|ICD10CM:E03.1|ICD9CM:2...,Uniprot_ID,Disease_Ontology,compiled,0.579,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30241575,Q8TDC2,ASSOCIATED_WITH_INTEGRATED,DOID:28,Protein,Disease,DISEASES,CKG,Serine/threonine-protein kinase BRSK1,endocrine system disease,ICD10CM:E34.9|ICD9CM:259.9|MESH:D004700|NCI:C3...,Uniprot_ID,Disease_Ontology,compiled,1.253,Generalised,Homo species
30241576,D3DVD9,ASSOCIATED_WITH_INTEGRATED,DOID:4305,Protein,Disease,DISEASES,CKG,C-reactive protein(1-205),bone giant cell tumor,MESH:D018212|NCI:C121932|SNOMEDCT_US_2023_03_0...,Uniprot_ID,Disease_Ontology,compiled,0.750,Generalised,Homo species
30241577,Q96EL3,ASSOCIATED_WITH_INTEGRATED,DOID:0080001,Protein,Disease,DISEASES,CKG,Large ribosomal subunit protein mL53 {ECO:0000...,bone disease,ICD10CM:M89.9|MESH:D001847|SNOMEDCT_US_2023_03...,Uniprot_ID,Disease_Ontology,compiled,0.572,Generalised,Homo species
30241578,P02788,ASSOCIATED_WITH_INTEGRATED,DOID:1657,Protein,Disease,DISEASES,CKG,Lactoferroxin-C,ventricular septal defect,GARD:7853|ICD10CM:Q21.0|ICD9CM:745.4|MESH:D006...,Uniprot_ID,Disease_Ontology,compiled,0.586,Generalised,Homo species


In [6]:
CKG_Protein_Disease2['kg_type'] = 'Generalised'
CKG_Protein_Disease2['kg_source'] = 'CKG'
CKG_Protein_Disease2['species'] = 'Homo species'
CKG_Protein_Disease2

,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,tail_do_alt_ids,head_id_is,tail_id_is,evidence_type,score,kg_type,species
0,P00747,ASSOCIATED_WITH,DOID:9970,Protein,Disease,DisGeNet: BEFREE,CKG,Plasmin light chain B,obesity,EFO:0001073|ICD10CM:E66.9|ICD9CM:278.00|MESH:D...,Uniprot_ID,Disease_Ontology,befree,0.10,Generalised,Homo species
1,Q13761,ASSOCIATED_WITH,DOID:1324,Protein,Disease,DisGeNet: BEFREE,CKG,Runt-related transcription factor 3,lung cancer,ICD10CM:C34.1|ICD10CM:C34.2|ICD10CM:C34.3|ICD9...,Uniprot_ID,Disease_Ontology,befree,0.06,Generalised,Homo species
2,Q96IZ0,ASSOCIATED_WITH,DOID:3869,Protein,Disease,DisGeNet: BEFREE,CKG,PRKC apoptosis WT1 regulator protein,childhood medulloblastoma,MESH:D008527|NCI:C3997|UMLS_CUI:C0278510,Uniprot_ID,Disease_Ontology,befree,0.02,Generalised,Homo species
3,Q04864,ASSOCIATED_WITH,DOID:4766,Protein,Disease,DisGeNet: BEFREE,CKG,Proto-oncogene c-Rel,embryoma,NCI:C8997|SNOMEDCT_US_2023_03_01:86049000|UMLS...,Uniprot_ID,Disease_Ontology,befree,0.05,Generalised,Homo species
4,P54259,ASSOCIATED_WITH,DOID:1441,Protein,Disease,DisGeNet: BEFREE,CKG,Atrophin-1,autosomal dominant cerebellar ataxia,MESH:D020754|MIM:PS164400|NCI:C82341|ORDO:94|S...,Uniprot_ID,Disease_Ontology,befree,0.09,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
927585,P09488,ASSOCIATED_WITH,DOID:0080409,Protein,Disease,DisGeNet: BEFREE,CKG,Glutathione S-transferase Mu 1 {ECO:0000305},familial adenomatous polyposis 1,MIM:175100,Uniprot_ID,Disease_Ontology,befree,0.01,Generalised,Homo species
927586,Q15436,ASSOCIATED_WITH,DOID:0050547,Protein,Disease,DisGeNet: BEFREE,CKG,Protein transport protein Sec23A {ECO:0000305},familial medullary thyroid carcinoma,MESH:C536911|MIM:155240,Uniprot_ID,Disease_Ontology,befree,0.01,Generalised,Homo species
927587,O95967,ASSOCIATED_WITH,DOID:0070138,Protein,Disease,DisGeNet: BEFREE,CKG,EGF-containing fibulin-like extracellular matr...,autosomal recessive cutis laxa type IIIB,ICD10CM:Q82.8|MIM:614438,Uniprot_ID,Disease_Ontology,befree,0.65,Generalised,Homo species
927588,Q00534,ASSOCIATED_WITH,DOID:7213,Protein,Disease,DisGeNet: BEFREE,CKG,Cyclin-dependent kinase 6,transitional meningioma,ICDO:9537/0|MESH:D008579|NCI:C4333|SNOMEDCT_US...,Uniprot_ID,Disease_Ontology,befree,0.01,Generalised,Homo species


### 2b. CrossBAR (×2)

The first file maps head ACs to RecNames at load time.

In [7]:
CrossBAR_Protein_Disease1 = pd.read_csv(PROC_DIR + 'crossbar/Protein_Disease_Crossbar.csv')
CrossBAR_Protein_Disease1.columns = CrossBAR_Protein_Disease1.columns.str.lower()
CrossBAR_Protein_Disease1['head_detail_name'] = CrossBAR_Protein_Disease1['head'].map(Uniprot_Recname_dict)

CrossBAR_Protein_Disease2 = pd.read_csv(PROC_DIR + 'crossbar/Protein_Disease_2_Crossbar.csv')
CrossBAR_Protein_Disease2.columns = CrossBAR_Protein_Disease2.columns.str.lower()

print(f"CrossBAR 1: {len(CrossBAR_Protein_Disease1):,} rows")
print(f"CrossBAR 2: {len(CrossBAR_Protein_Disease2):,} rows")

CrossBAR 1: 92 rows
CrossBAR 2: 3 rows


In [8]:
CrossBAR_Protein_Disease1['kg_type'] = 'Generalised'
CrossBAR_Protein_Disease1['kg_source'] = 'CROssBAR'
CrossBAR_Protein_Disease1['species'] = 'Homo species'
CrossBAR_Protein_Disease1

,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,tail_alt_id,tail_detail_name,head_id_is,tail_id_is,head_detail_name,kg_type,species
0,O75027,Protein_Disease,DOID:0050554,Protein,Disease,CROssBAR,ABCB7,Orphanet:2802,X-linked sideroblastic anemia with ataxia,Uniprot_ID,DOID,"Iron-sulfur clusters transporter ABCB7, mitoch...",Generalised,Homo species
1,O76024,Protein_Disease,DOID:10632,Protein,Disease,CROssBAR,WFS1,Orphanet:3463,Wolfram syndrome,Uniprot_ID,DOID,Wolframin,Generalised,Homo species
2,O75161,Protein_Disease,DOID:0050576,Protein,Disease,CROssBAR,NPHP4,Orphanet:3156,Senior-Loken syndrome,Uniprot_ID,DOID,Nephrocystin-4,Generalised,Homo species
3,O15072,Protein_Disease,DOID:0060366,Protein,Disease,CROssBAR,ADAMTS3,Orphanet:2136,Hennekam syndrome,Uniprot_ID,DOID,A disintegrin and metalloproteinase with throm...,Generalised,Homo species
4,O75487,Protein_Disease,DOID:0111842,Protein,Disease,CROssBAR,GPC4,Orphanet:2662,Keipert syndrome,Uniprot_ID,DOID,Secreted glypican-4,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,O60216,Protein_Disease,DOID:11725,Protein,Disease,CROssBAR,RAD21,Orphanet:199,Cornelia de Lange syndrome,Uniprot_ID,DOID,64-kDa C-terminal product {ECO:0000303|PubMed:...,Generalised,Homo species
88,A8MTZ0,Protein_Disease,DOID:1935,Protein,Disease,CROssBAR,BBIP1,Orphanet:110,Bardet-Biedl syndrome,Uniprot_ID,DOID,BBSome-interacting protein 1,Generalised,Homo species
89,O15360,Protein_Disease,DOID:13636,Protein,Disease,CROssBAR,FANCA,Orphanet:84,Fanconi anemia,Uniprot_ID,DOID,Fanconi anemia group A protein,Generalised,Homo species
90,O15550,Protein_Disease,DOID:0060473,Protein,Disease,CROssBAR,KDM6A,Orphanet:2322,Kabuki syndrome,Uniprot_ID,DOID,Lysine-specific demethylase 6A,Generalised,Homo species


In [9]:
CrossBAR_Protein_Disease2['kg_type'] = 'Generalised'
CrossBAR_Protein_Disease2['kg_source'] = 'CROssBAR'
CrossBAR_Protein_Disease2['species'] = 'Homo species'
CrossBAR_Protein_Disease2

,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,B2CW77,Protein_Disease,DOID:6457,Protein,Disease,CROssBAR,KLLN,Killin,Cowden syndrome,Uniprot_ID,DOID,Generalised,Homo species
1,A6NI61,Protein_Disease,DOID:0080194,Protein,Disease,CROssBAR,MYMK,Protein myomaker {ECO:0000250|UniProtKB:Q9D1N4},Carey-Fineman-Ziter syndrome,Uniprot_ID,DOID,Generalised,Homo species
2,A8MTZ0,Protein_Disease,DOID:1935,Protein,Disease,CROssBAR,BBIP1,BBSome-interacting protein 1,Bardet-Biedl syndrome,Uniprot_ID,DOID,Generalised,Homo species


### 2c. DtiNet

In [10]:
DtiNet_Protein_Disease = pd.read_csv(PROC_DIR + 'DTINet/Protein_Disease_DTINet.csv')
DtiNet_Protein_Disease.columns = DtiNet_Protein_Disease.columns.str.lower()
print(f"DtiNet:     {len(DtiNet_Protein_Disease):,} rows")
DtiNet_Protein_Disease.head(2)

DtiNet:     766,041 rows


,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,head_detail_name,tail_detail_name,head_id_is,tail_id_is
0,Q9UI32,Protein_Disease,DOID:1596,Protein,Disease,DTInet,Q9UI32,"Glutaminase liver isoform, mitochondrial",depressive disorder,Uniprot_ID,DOID
1,Q9UI32,Protein_Disease,C0860207,Protein,Disease,DTInet,Q9UI32,"Glutaminase liver isoform, mitochondrial",drug-induced liver injury,Uniprot_ID,MESH


In [11]:
DtiNet_Protein_Disease['kg_type'] = 'Generalised'
DtiNet_Protein_Disease['kg_source'] = 'DtiNet'
DtiNet_Protein_Disease['species'] = 'Homo species'
DtiNet_Protein_Disease

,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,Q9UI32,Protein_Disease,DOID:1596,Protein,Disease,DtiNet,Q9UI32,"Glutaminase liver isoform, mitochondrial",depressive disorder,Uniprot_ID,DOID,Generalised,Homo species
1,Q9UI32,Protein_Disease,C0860207,Protein,Disease,DtiNet,Q9UI32,"Glutaminase liver isoform, mitochondrial",drug-induced liver injury,Uniprot_ID,MESH,Generalised,Homo species
2,Q9UI32,Protein_Disease,C0027540,Protein,Disease,DtiNet,Q9UI32,"Glutaminase liver isoform, mitochondrial",necrosis,Uniprot_ID,MESH,Generalised,Homo species
3,Q9UI32,Protein_Disease,DOID:12849,Protein,Disease,DtiNet,Q9UI32,"Glutaminase liver isoform, mitochondrial",autistic disorder,Uniprot_ID,DOID,Generalised,Homo species
4,Q9UI32,Protein_Disease,DOID:1679,Protein,Disease,DtiNet,Q9UI32,"Glutaminase liver isoform, mitochondrial",cystitis,Uniprot_ID,DOID,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
766036,P0C0L5,Protein_Disease,DOID:3275,Protein,Disease,DtiNet,P0C0L5,Complement C4 gamma chain,thymoma,Uniprot_ID,DOID,Generalised,Homo species
766037,P0C0L5,Protein_Disease,DOID:9266,Protein,Disease,DtiNet,P0C0L5,Complement C4 gamma chain,cystinuria,Uniprot_ID,DOID,Generalised,Homo species
766038,P0C0L5,Protein_Disease,DOID:2862,Protein,Disease,DtiNet,P0C0L5,Complement C4 gamma chain,glucosephosphate dehydrogenase deficiency,Uniprot_ID,DOID,Generalised,Homo species
766039,P0C0L5,Protein_Disease,DOID:801,Protein,Disease,DtiNet,P0C0L5,Complement C4 gamma chain,hemarthrosis,Uniprot_ID,DOID,Generalised,Homo species


## 3. Check and Fix Duplicate Columns

In [12]:
SOURCE_DFS = [
    ('CKG_Protein_Disease1',      CKG_Protein_Disease1),
    ('CKG_Protein_Disease2',      CKG_Protein_Disease2),
    ('CrossBAR_Protein_Disease1', CrossBAR_Protein_Disease1),
    ('CrossBAR_Protein_Disease2', CrossBAR_Protein_Disease2),
    ('DtiNet_Protein_Disease',    DtiNet_Protein_Disease),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[CKG_Protein_Disease1] ✓ no duplicates
[CKG_Protein_Disease2] ✓ no duplicates
[CrossBAR_Protein_Disease1] ✓ no duplicates
[CrossBAR_Protein_Disease2] ✓ no duplicates
[DtiNet_Protein_Disease] ✓ no duplicates


In [13]:
CKG_Protein_Disease1      = CKG_Protein_Disease1.loc[:,      ~CKG_Protein_Disease1.columns.duplicated(keep='first')]
CKG_Protein_Disease2      = CKG_Protein_Disease2.loc[:,      ~CKG_Protein_Disease2.columns.duplicated(keep='first')]
CrossBAR_Protein_Disease1 = CrossBAR_Protein_Disease1.loc[:, ~CrossBAR_Protein_Disease1.columns.duplicated(keep='first')]
CrossBAR_Protein_Disease2 = CrossBAR_Protein_Disease2.loc[:, ~CrossBAR_Protein_Disease2.columns.duplicated(keep='first')]
DtiNet_Protein_Disease    = DtiNet_Protein_Disease.loc[:,    ~DtiNet_Protein_Disease.columns.duplicated(keep='first')]

for name, df in SOURCE_DFS:
    remaining = df.columns[df.columns.duplicated()].tolist()
    print(f"[{name}] {'✓ clean' if not remaining else '✗ dupes: ' + str(remaining)}")

[CKG_Protein_Disease1] ✓ clean
[CKG_Protein_Disease2] ✓ clean
[CrossBAR_Protein_Disease1] ✓ clean
[CrossBAR_Protein_Disease2] ✓ clean
[DtiNet_Protein_Disease] ✓ clean


## 4. Align Schemas and Concatenate

In [14]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name','kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    tmp['head_id_is'] = tmp['head_id_is'].astype(str)
    tmp['tail_id_is'] = tmp['tail_id_is'].astype(str)
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 31,935,306 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,Q9UCA1,ASSOCIATED_WITH_INTEGRATED,DOID:10605,Protein,NaN,Disease,CKG,Uniprot_ID,Disease_Ontology,Thrombin heavy chain,short bowel syndrome,Generalised,Homo species
1,Q8N425,ASSOCIATED_WITH_INTEGRATED,DOID:654,Protein,NaN,Disease,CKG,Uniprot_ID,Disease_Ontology,(E3-independent) E2 ubiquitin-conjugating enzyme,overnutrition,Generalised,Homo species


## 5. Standardise Fixed-Value Columns

In [15]:
final_df['head_type'] = 'Protein'
final_df['tail_type'] = 'Disease'
final_df['relation']  = 'Protein_Disease'

# tail_id_is: DOID-prefix → DOID, else MESH_ID
final_df['tail'] = final_df['tail'].astype(str)
final_df['tail_id_is'] = np.where(
    final_df['tail'].isna(), None,
    np.where(final_df['tail'].str.startswith('DOID'), 'DOID', 'MESH_ID')
)

print("Unique relation:",   set(final_df['relation']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'Protein_Disease'}
Unique tail_id_is: {'DOID', 'MESH_ID'}
Unique kg_source: {'CKG', 'CROssBAR', 'DtiNet'}


## 6. Deduplicate

In [16]:
GROUP_COLS = ['head', 'relation', 'tail']

FIRST_COLS = ['head_type', 'relation_type', 'tail_type',
              'head_id_is', 'tail_id_is',
              'head_detail_name', 'tail_detail_name', 'species']
MERGE_COLS = ['kg_source', 'kg_type']

# ── 1) Fast path for all the 'first' columns ──────────────────────────────────
# groupby with built-in 'first' is C-optimized (no Python callback per group)
first_part = (
    final_df
    .groupby(GROUP_COLS, sort=False, as_index=False)[FIRST_COLS]
    .first()
)

# ── 2) Merge columns: build '::'-joined sorted-unique strings, vectorized ─────
def fast_merge(df, col):
    s = df[[*GROUP_COLS, col]].dropna(subset=[col])
    # drop duplicate (group, value) pairs so each value appears once per group
    s = s.drop_duplicates()
    # sort so the join is deterministic (matches sorted(set(...)))
    s = s.sort_values(col)
    out = (
        s.groupby(GROUP_COLS, sort=False)[col]
         .agg('::'.join)
         .reset_index()
    )
    return out

merge_parts = first_part
for col in MERGE_COLS:
    merge_parts = merge_parts.merge(fast_merge(final_df, col),
                                    on=GROUP_COLS, how='left')

final_df_dedup = merge_parts
print(f"After deduplication: {len(final_df_dedup):,} rows")
final_df_dedup

After deduplication: 31,250,376 rows


,head,relation,tail,head_type,relation_type,tail_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species,kg_source,kg_type
0,Q9UCA1,Protein_Disease,DOID:10605,Protein,None,Disease,Uniprot_ID,DOID,Thrombin heavy chain,short bowel syndrome,Homo species,CKG,Generalised
1,Q8N425,Protein_Disease,DOID:654,Protein,None,Disease,Uniprot_ID,DOID,(E3-independent) E2 ubiquitin-conjugating enzyme,overnutrition,Homo species,CKG,Generalised
2,Q86UJ9,Protein_Disease,DOID:0060037,Protein,None,Disease,Uniprot_ID,DOID,Cyclic AMP-responsive element-binding protein 5,developmental disorder of mental health,Homo species,CKG,Generalised
3,Q7Z639,Protein_Disease,DOID:1240,Protein,None,Disease,Uniprot_ID,DOID,Ubiquitin conjugation factor E4 A {ECO:0000305},leukemia,Homo species,CKG,Generalised
4,Q1P9S9,Protein_Disease,DOID:0050328,Protein,None,Disease,Uniprot_ID,DOID,Arresten,congenital hypothyroidism,Homo species,CKG,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...,...
31250371,P0C0L5,Protein_Disease,DOID:3275,Protein,None,Disease,Uniprot_ID,DOID,Complement C4 gamma chain,thymoma,Homo species,DtiNet,Generalised
31250372,P0C0L5,Protein_Disease,DOID:9266,Protein,None,Disease,Uniprot_ID,DOID,Complement C4 gamma chain,cystinuria,Homo species,DtiNet,Generalised
31250373,P0C0L5,Protein_Disease,DOID:2862,Protein,None,Disease,Uniprot_ID,DOID,Complement C4 gamma chain,glucosephosphate dehydrogenase deficiency,Homo species,DtiNet,Generalised
31250374,P0C0L5,Protein_Disease,DOID:801,Protein,None,Disease,Uniprot_ID,DOID,Complement C4 gamma chain,hemarthrosis,Homo species,DtiNet,Generalised


In [17]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,31250376,0,31250376
tail_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,53475,0,53475
tail_detail_name,2425,0,2425


## 7. Fill Missing Protein Head Names and Drop Unresolved Rows

In [18]:
# Normalise fake NaNs, fill head_detail_name from UniProt TrEMBL dict, drop remaining
final_df_dedup['head_detail_name'] = final_df_dedup['head_detail_name'].replace(['NAN', 'NaN', 'None'], np.nan)
final_df_dedup['head_detail_name'] = final_df_dedup['head_detail_name'].fillna(
    final_df_dedup['head'].map(uniprot_trembl_AC_Name_dict)
)

before = len(final_df_dedup)
final_df_dedup = final_df_dedup[~final_df_dedup['head_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} rows with missing head_detail_name → {len(final_df_dedup):,} remaining")

# Also drop rows with no tail_detail_name
before = len(final_df_dedup)
final_df_dedup = final_df_dedup[~final_df_dedup['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} rows with missing tail_detail_name → {len(final_df_dedup):,} remaining")

Dropped 5,783 rows with missing head_detail_name → 31,244,593 remaining
Dropped 2,425 rows with missing tail_detail_name → 31,242,168 remaining


## 8. Add Schema Columns and Enforce Column Order

In [19]:
final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (31242168, 13)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,Q9UCA1,Protein_Disease,DOID:10605,Protein,None,Disease,CKG,Uniprot_ID,DOID,Thrombin heavy chain,short bowel syndrome,Generalised,Homo species
1,Q8N425,Protein_Disease,DOID:654,Protein,None,Disease,CKG,Uniprot_ID,DOID,(E3-independent) E2 ubiquitin-conjugating enzyme,overnutrition,Generalised,Homo species
2,Q86UJ9,Protein_Disease,DOID:0060037,Protein,None,Disease,CKG,Uniprot_ID,DOID,Cyclic AMP-responsive element-binding protein 5,developmental disorder of mental health,Generalised,Homo species
3,Q7Z639,Protein_Disease,DOID:1240,Protein,None,Disease,CKG,Uniprot_ID,DOID,Ubiquitin conjugation factor E4 A {ECO:0000305},leukemia,Generalised,Homo species
4,Q1P9S9,Protein_Disease,DOID:0050328,Protein,None,Disease,CKG,Uniprot_ID,DOID,Arresten,congenital hypothyroidism,Generalised,Homo species


## 9. QC — NaN Check

In [20]:
nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,31242168,0,31242168
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [21]:
final_df_dedup['species'] = 'Homo sapiens'
final_df_dedup['kg_type'] = 'Generalised'

## 10. Save Output

In [22]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 31,242,168 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PROTEIN_DISEASE/ALL_PROTEIN_DISEASE.parquet


# Final Mapping

In [23]:
# =========================================================
# LOAD DISEASE MAPPING FILE
# =========================================================

DB_DIR = BASE_DIR + 'data_collection/databases_for_mapping/'

final_file_CH_D = pd.read_csv(
    DB_DIR + 'DO/ultimate_DISEASE_dictionary.csv'
)

# =========================================================
# PRIORITIZE DOID IDs
# =========================================================

# rows having DOID
doid_rows = final_file_CH_D[
    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)
]

# entity names that have at least one DOID
names_with_doid = set(
    doid_rows['entity_name']
    .str.lower()
    .str.strip()
)

# keep:
# 1. all DOID rows
# 2. non-DOID rows only if no DOID exists

final_diseaseid_df = final_file_CH_D[

    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)

    |

    ~final_file_CH_D['entity_name']
    .str.lower()
    .str.strip()
    .isin(names_with_doid)
]

final_diseaseid_df = (
    final_diseaseid_df
    .reset_index(drop=True)
)

# =========================================================
# CREATE DISEASE NAME -> ID DICTIONARY
# =========================================================

disease_dict = dict(

    zip(

        final_diseaseid_df['entity_name']
        .str.lower()
        .str.strip(),

        final_diseaseid_df['entity_id']
    )
)

# =========================================================
# UNIVERSAL ENTITY MAPPING FUNCTION
# =========================================================

def map_entity_ids(
    df,
    mapping_dict,
    side='tail'
):
    """
    Universal KG entity mapper.

    Parameters
    ----------
    df : pd.DataFrame

    mapping_dict : dict
        entity_name -> entity_id

    side : str
        'head' or 'tail'

    Returns
    -------
    pd.DataFrame
    """

    # -----------------------------------------
    # dynamic column names
    # -----------------------------------------

    detail_col = f'{side}_detail_name'

    id_col = side

    id_is_col = f'{side}_id_is'

    # -----------------------------------------
    # map IDs
    # -----------------------------------------

    df[id_col] = (df[detail_col].astype(str).str.lower().str.strip().map(mapping_dict))

    # -----------------------------------------
    # assign ID source
    # -----------------------------------------

    df[id_is_col] = None

    # DOID
    # DOID
    df.loc[
        df[id_col].str.startswith('DOID:', na=False),id_is_col] = 'DOID'
    
    # everything else -> MESH
    df.loc[~df[id_col].str.startswith('DOID:', na=False)&df[id_col].notna(),id_is_col] = 'MESH'

    return df
    

In [24]:
final_file = pd.read_parquet(OUT_PATH)
final_file

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,Q9UCA1,Protein_Disease,DOID:10605,Protein,None,Disease,CKG,Uniprot_ID,DOID,Thrombin heavy chain,short bowel syndrome,Generalised,Homo sapiens
1,Q8N425,Protein_Disease,DOID:654,Protein,None,Disease,CKG,Uniprot_ID,DOID,(E3-independent) E2 ubiquitin-conjugating enzyme,overnutrition,Generalised,Homo sapiens
2,Q86UJ9,Protein_Disease,DOID:0060037,Protein,None,Disease,CKG,Uniprot_ID,DOID,Cyclic AMP-responsive element-binding protein 5,developmental disorder of mental health,Generalised,Homo sapiens
3,Q7Z639,Protein_Disease,DOID:1240,Protein,None,Disease,CKG,Uniprot_ID,DOID,Ubiquitin conjugation factor E4 A {ECO:0000305},leukemia,Generalised,Homo sapiens
4,Q1P9S9,Protein_Disease,DOID:0050328,Protein,None,Disease,CKG,Uniprot_ID,DOID,Arresten,congenital hypothyroidism,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
31242163,P0C0L5,Protein_Disease,DOID:3275,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,thymoma,Generalised,Homo sapiens
31242164,P0C0L5,Protein_Disease,DOID:9266,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,cystinuria,Generalised,Homo sapiens
31242165,P0C0L5,Protein_Disease,DOID:2862,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,glucosephosphate dehydrogenase deficiency,Generalised,Homo sapiens
31242166,P0C0L5,Protein_Disease,DOID:801,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,hemarthrosis,Generalised,Homo sapiens


In [25]:

final_file = map_entity_ids(df=final_file,mapping_dict=disease_dict,side='tail')
final_file

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,Q9UCA1,Protein_Disease,DOID:10605,Protein,None,Disease,CKG,Uniprot_ID,DOID,Thrombin heavy chain,short bowel syndrome,Generalised,Homo sapiens
1,Q8N425,Protein_Disease,DOID:654,Protein,None,Disease,CKG,Uniprot_ID,DOID,(E3-independent) E2 ubiquitin-conjugating enzyme,overnutrition,Generalised,Homo sapiens
2,Q86UJ9,Protein_Disease,DOID:0060037,Protein,None,Disease,CKG,Uniprot_ID,DOID,Cyclic AMP-responsive element-binding protein 5,developmental disorder of mental health,Generalised,Homo sapiens
3,Q7Z639,Protein_Disease,DOID:1240,Protein,None,Disease,CKG,Uniprot_ID,DOID,Ubiquitin conjugation factor E4 A {ECO:0000305},leukemia,Generalised,Homo sapiens
4,Q1P9S9,Protein_Disease,DOID:0050328,Protein,None,Disease,CKG,Uniprot_ID,DOID,Arresten,congenital hypothyroidism,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
31242163,P0C0L5,Protein_Disease,DOID:3275,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,thymoma,Generalised,Homo sapiens
31242164,P0C0L5,Protein_Disease,DOID:9266,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,cystinuria,Generalised,Homo sapiens
31242165,P0C0L5,Protein_Disease,DOID:2862,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,glucosephosphate dehydrogenase deficiency,Generalised,Homo sapiens
31242166,P0C0L5,Protein_Disease,DOID:801,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,hemarthrosis,Generalised,Homo sapiens


In [26]:
final_file = final_file[~final_file['tail'].isna()]
final_file

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,Q9UCA1,Protein_Disease,DOID:10605,Protein,None,Disease,CKG,Uniprot_ID,DOID,Thrombin heavy chain,short bowel syndrome,Generalised,Homo sapiens
1,Q8N425,Protein_Disease,DOID:654,Protein,None,Disease,CKG,Uniprot_ID,DOID,(E3-independent) E2 ubiquitin-conjugating enzyme,overnutrition,Generalised,Homo sapiens
2,Q86UJ9,Protein_Disease,DOID:0060037,Protein,None,Disease,CKG,Uniprot_ID,DOID,Cyclic AMP-responsive element-binding protein 5,developmental disorder of mental health,Generalised,Homo sapiens
3,Q7Z639,Protein_Disease,DOID:1240,Protein,None,Disease,CKG,Uniprot_ID,DOID,Ubiquitin conjugation factor E4 A {ECO:0000305},leukemia,Generalised,Homo sapiens
4,Q1P9S9,Protein_Disease,DOID:0050328,Protein,None,Disease,CKG,Uniprot_ID,DOID,Arresten,congenital hypothyroidism,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
31242163,P0C0L5,Protein_Disease,DOID:3275,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,thymoma,Generalised,Homo sapiens
31242164,P0C0L5,Protein_Disease,DOID:9266,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,cystinuria,Generalised,Homo sapiens
31242165,P0C0L5,Protein_Disease,DOID:2862,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,glucosephosphate dehydrogenase deficiency,Generalised,Homo sapiens
31242166,P0C0L5,Protein_Disease,DOID:801,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,hemarthrosis,Generalised,Homo sapiens


In [30]:
final_file.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(final_file):,} rows → {OUT_PATH}")

Saved 31,233,883 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PROTEIN_DISEASE/ALL_PROTEIN_DISEASE.parquet


In [31]:
# pd.read_parquet('/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PROTEIN_DISEASE/ALL_PROTEIN_DISEASE.parquet')

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,Q9UCA1,Protein_Disease,DOID:10605,Protein,None,Disease,CKG,Uniprot_ID,DOID,Thrombin heavy chain,short bowel syndrome,Generalised,Homo sapiens
1,Q8N425,Protein_Disease,DOID:654,Protein,None,Disease,CKG,Uniprot_ID,DOID,(E3-independent) E2 ubiquitin-conjugating enzyme,overnutrition,Generalised,Homo sapiens
2,Q86UJ9,Protein_Disease,DOID:0060037,Protein,None,Disease,CKG,Uniprot_ID,DOID,Cyclic AMP-responsive element-binding protein 5,developmental disorder of mental health,Generalised,Homo sapiens
3,Q7Z639,Protein_Disease,DOID:1240,Protein,None,Disease,CKG,Uniprot_ID,DOID,Ubiquitin conjugation factor E4 A {ECO:0000305},leukemia,Generalised,Homo sapiens
4,Q1P9S9,Protein_Disease,DOID:0050328,Protein,None,Disease,CKG,Uniprot_ID,DOID,Arresten,congenital hypothyroidism,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
31233878,P0C0L5,Protein_Disease,DOID:3275,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,thymoma,Generalised,Homo sapiens
31233879,P0C0L5,Protein_Disease,DOID:9266,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,cystinuria,Generalised,Homo sapiens
31233880,P0C0L5,Protein_Disease,DOID:2862,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,glucosephosphate dehydrogenase deficiency,Generalised,Homo sapiens
31233881,P0C0L5,Protein_Disease,DOID:801,Protein,None,Disease,DtiNet,Uniprot_ID,DOID,Complement C4 gamma chain,hemarthrosis,Generalised,Homo sapiens
